In [1]:
import pandas as pd 
import numpy as np 
import os
import datetime


In [2]:
df = pd.read_csv('/data/7TB/nick/ai_readi_v3/tabular_processed_data/synthetic_rescaled.csv', index_col=0)
#df.to_csv('/data/7TB/nick/synth_ai_readi_tabular/synthetic_data_all.tsv', sep='\t')

# Create Directories

In [3]:
base = '/data/7TB/nick/synth_ai_readi_tabular'

# Top level directory
os.makedirs(base, exist_ok=True)

# Modality-specific directories
os.makedirs(os.path.join(base, 'clinical_data'), exist_ok=True)
os.makedirs(os.path.join(base, 'cardiac_ecg'), exist_ok=True)
os.makedirs(os.path.join(base, 'wearable_activity_monitor'), exist_ok=True)
os.makedirs(os.path.join(base, 'wearable_blood_glucose'), exist_ok=True)

# Clinical Data

## Conditions

In [4]:
real = pd.read_csv('/data/7TB/nick/ai_readi_v3/clinical_data/condition_occurrence.csv', index_col=None)
pat = pd.read_csv('/data/7TB/nick/ai_readi_v3/participants.tsv', delimiter='\t').rename(columns={'participant_id':'person_id'})

In [5]:
df.columns.tolist()

['heart_rate_mean',
 'blood_glucose_mean',
 'resp_rate_mean',
 'stress_mean',
 'blood_glucose_std',
 'total_steps',
 'resting_heart_rate',
 'total_kcal',
 'sleep_light_hrs',
 'sleep_deep_hrs',
 'sleep_rem_hrs',
 'sleep_awake_hrs',
 'act_generic_hrs',
 'act_running_hrs',
 'act_walking_hrs',
 'act_sedentary_hrs',
 'BMI',
 'Diastolic (mmHg)',
 'Systolic (mmHg)',
 'clock_visuospatial_executive',
 'clock_visuospatial_executive_time',
 'cube_visuospatial_executive',
 'cube_visuospatial_executive_time',
 'delayed_recall_with_no_clue',
 'delayed_recall_with_no_clue_time',
 'digitspan',
 'digitspan_time',
 'Height (cm)',
 'Hip Circumference (cm)',
 'Albumin/Globulin ratio',
 'Albumin [Mass/volume] in Serum or_value (g/dL)',
 'Alkaline phosphatase_value (IU/L)',
 'Alanine aminotransferase [Enzymat_value (IU/L)',
 'Aspartate aminotransferase [Enzym_value (IU/L)',
 'Bilirubin.total [Mass/vol',
 'Urea nitrogen [Mass/volume] in Serum _value (mg/dL)',
 'BUN/Creatinine ratio',
 'C peptide [Mass/volume

In [6]:
cond_cols = ['Age-related macular degeneration', 'Arthritis', 'Cancer', 'Cataracts (1+ eyes)', 'Chronic pullmonary problems', 'Circulation problems', 'Dementia/Alzheimers', 'Diabetic retinopathy (1+)', 
            'Digestive problems', 'Dry eye (1+)', 'Elevated A1C', 'Glaucoma (1+)', 'Hearing impairment', 'Heart attack', 'High blood cholesterol', 'High blood pressure', 'Kidney problems', 'Low blood pressure', 
            'Mild cognitive impairmen', 'Multiple sclerosis', 'Obesity', 'Osteoporosis', 'Other heart issues (pacemaker)', 'Other neurological conditions', "Parkinson's disease", 'Pre-diabetes', 'Retinal vascular occlusion', 
            'Stroke', 'Type 2 Diabetes', 'Urinary problems']

In [7]:
names = np.array(real['condition_source_value'].value_counts().index.tolist())


mapping = { 
        'Age-related macular degeneration (AM': "Age-related macular degeneration", 
        'Arthritis (joint pain)': 'Arthritis', 
        'Cancer (any type)': 'Cancer',
        'Cataracts (in one or both eyes)': 'Cataracts (1+ eyes)',
        'Chronic pulmonary (lung) problems (E': 'Chronic pullmonary problems',
        'Circulation problems (Examples: art': 'Circulation problems',
        "Dementia (Examples: Alzheimer's Disea": 'Dementia/Alzheimers',
        'Diabetic retinopathy (in one or both': 'Diabetic retinopathy (1+)',
        'Digestive problems (Examples: stomach': 'Digestive problems', 
        'Dry eye (in one or both eyes)': 'Dry eye (1+)',
        'Elevated A1C levels (elevated blood sugar': 'Elevated A1C',
        'Glaucoma (in one or both eyes)': 'Glaucoma (1+)', 
        'Mild cognitive impairment (known as': 'Mild cognitive impairmen',
        'Other heart issues (Examples: pace': 'Other heart issues (pacemaker)',
        'Retinal vascular occlusion ("stroke': 'Retinal vascular occlusion', 
        'Urinary problems (Examples: urinary t': 'Urinary problems', 
        'Type II Diabetes': 'Type 2 Diabetes'}

reverse_mapping = {v:k for k,v in mapping.items()}

In [8]:

cols_new_df = []
counter = 0

for col_og in cond_cols:
    try:
        column = names[np.array([reverse_mapping[col_og] in y for y in names])][0]
    except: 
        column = names[np.array([col_og in y for y in names])][0]
    
    if not column in names:
        raise StopIteration("Cannot find correct name for conditions column")
    
    print(column)

    temp = real[real['condition_source_value'] == column]


    cols_same = {col:[] for col in temp.columns if len(temp[col].value_counts()) == 1 and not temp[col].isnull().mean() == 1}
    cols_nan = {col:[] for col in temp.columns if temp[col].isnull().mean() == 1}
    cols_new = {col:[] for col in temp.columns if len(temp[col].value_counts()) != 1 and not temp[col].isnull().mean() == 1}

    for i, row in df.iterrows():

        if row[col_og] == 1:

            for col in cols_same:
                cols_same[col].append(temp.iloc[0][col])

            for col in cols_nan:
                cols_nan[col].append(np.nan)

            cols_new['condition_occurrence_id'].append(int(counter))
            cols_new['person_id'].append(int(row.name))

            now = datetime.datetime.now()
            omop_date = now.date()                     # date: YYYY-MM-DD
            omop_datetime = now.strftime('%Y-%m-%d %H:%M:%S')  # datetime: string format

            cols_new['condition_start_date'].append(omop_date)
            cols_new['condition_end_date'].append(omop_date)
            cols_new['condition_start_datetime'].append(omop_datetime)
            cols_new['condition_end_datetime'].append(omop_datetime)
            cols_new['visit_occurrence_id'].append(int(row.name))

            counter += 1
    merged = {**cols_new, **cols_same, **cols_nan}
    cols_new_df.append(pd.DataFrame(merged))



mhoccur_amd, Age-related macular degeneration (AM
mhoccur_ra, Arthritis (joint pain)
mhoccur_ca, Cancer (any type)
mhoccur_crt, Cataracts (in one or both eyes)
mhoccur_plm, Chronic pulmonary (lung) problems (E
mhoccur_circ, Circulation problems (Examples: art
mhoccur_ad, Dementia (Examples: Alzheimer's Disea
mhoccur_pdr, Diabetic retinopathy (in one or both
mhoccur_gi, Digestive problems (Examples: stomach
mhoccur_ded, Dry eye (in one or both eyes)
mh_a1c, Elevated A1C levels (elevated blood sugar
mhoccur_glc, Glaucoma (in one or both eyes)
mhoccur_ear, Hearing impairment
mhoccur_mi, Heart attack
mhoccur_clsh, High blood cholesterol
mhoccur_hbp, High blood pressure
mhoccur_rnl, Kidney problems
mhoccur_lbp, Low blood pressure
mhoccur_cogn, Mild cognitive impairment (known as
mhoccur_ms, Multiple sclerosis
mhoccur_obs, Obesity
mhoccur_oa, Osteoporosis
mhoccur_cvdot, Other heart issues (Examples: pace
mhoccur_cns, Other neurological conditions
mhoccur_pd, Parkinson's disease
mhterm_predm,

In [9]:
conditions_new = pd.concat(cols_new_df)[real.columns]
conditions_new.to_csv('/data/7TB/nick/synth_ai_readi_tabular/clinical_data/condition_occurrence.csv')
conditions_new

,condition_occurrence_id,person_id,condition_concept_id,condition_start_date,condition_start_datetime,condition_end_date,condition_end_datetime,condition_type_concept_id,condition_status_concept_id,stop_reason,provider_id,visit_occurrence_id,visit_detail_id,condition_source_value,condition_source_concept_id,condition_status_source_value
0,0,2,374028,2026-01-15,2026-01-15 14:16:01,2026-01-15,2026-01-15 14:16:01,45905770,32893,,0,2,0,"mhoccur_amd, Age-related macular degeneration (AM",0,
1,1,19,374028,2026-01-15,2026-01-15 14:16:01,2026-01-15,2026-01-15 14:16:01,45905770,32893,,0,19,0,"mhoccur_amd, Age-related macular degeneration (AM",0,
2,2,29,374028,2026-01-15,2026-01-15 14:16:01,2026-01-15,2026-01-15 14:16:01,45905770,32893,,0,29,0,"mhoccur_amd, Age-related macular degeneration (AM",0,
3,3,33,374028,2026-01-15,2026-01-15 14:16:01,2026-01-15,2026-01-15 14:16:01,45905770,32893,,0,33,0,"mhoccur_amd, Age-related macular degeneration (AM",0,
4,4,44,374028,2026-01-15,2026-01-15 14:16:01,2026-01-15,2026-01-15 14:16:01,45905770,32893,,0,44,0,"mhoccur_amd, Age-related macular degeneration (AM",0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2453,59164,10502,81902,2026-01-15,2026-01-15 14:16:13,2026-01-15,2026-01-15 14:16:13,45905770,32893,,0,10502,0,"mhoccur_ua, Urinary problems (Examples: urinary t",0,
2454,59165,10504,81902,2026-01-15,2026-01-15 14:16:13,2026-01-15,2026-01-15 14:16:13,45905770,32893,,0,10504,0,"mhoccur_ua, Urinary problems (Examples: urinary t",0,
2455,59166,10506,81902,2026-01-15,2026-01-15 14:16:13,2026-01-15,2026-01-15 14:16:13,45905770,32893,,0,10506,0,"mhoccur_ua, Urinary problems (Examples: urinary t",0,
2456,59167,10510,81902,2026-01-15,2026-01-15 14:16:13,2026-01-15,2026-01-15 14:16:13,45905770,32893,,0,10510,0,"mhoccur_ua, Urinary problems (Examples: urinary t",0,


## Measurements

In [10]:
real = pd.read_csv('/data/7TB/nick/ai_readi_v3/clinical_data/measurement.csv', index_col=0)
pat = pd.read_csv('/data/7TB/nick/ai_readi_v3/participants.tsv', delimiter='\t').rename(columns={'participant_id':'person_id'})

/tmp/ipykernel_3960453/3841819678.py:1: DtypeWarning: Columns (19,25) have mixed types. Specify dtype option on import or set low_memory=False.
  real = pd.read_csv('/data/7TB/nick/ai_readi_v3/clinical_data/measurement.csv', index_col=0)


In [11]:
meas_cols = ['BMI', 'Diastolic (mmHg)', 'Systolic (mmHg)', 'clock_visuospatial_executive', 'clock_visuospatial_executive_time', 'cube_visuospatial_executive', 'cube_visuospatial_executive_time', 'delayed_recall_with_no_clue', 'delayed_recall_with_no_clue_time', 'digitspan', 'digitspan_time', 
        'Height (cm)', 'Hip Circumference (cm)', 'Albumin/Globulin ratio', 'Albumin [Mass/volume] in Serum or_value (g/dL)', 'Alkaline phosphatase_value (IU/L)', 'Alanine aminotransferase [Enzymat_value (IU/L)', 'Aspartate aminotransferase [Enzym_value (IU/L)',
        'Bilirubin.total [Mass/vol', 'Urea nitrogen [Mass/volume] in Serum _value (mg/dL)', 'BUN/Creatinine ratio', 'C peptide [Mass/volume] in Seru_value (ng/L)', 'Calcium [Mass/volume] in Serum or_value (mEq/L)', 'Carbon dioxide', 'Chloride [Moles/volume] in Serum_value (mEq/L)', 
        'Creatinine [Mass/volume] in Se_value (mg/dL)', 'C reactive protein [Mass/volume] i_value (mg/L)', 'Globulin [Mass/volume] in ', 'Glucose [Mass/volume] in Serum or_value (mg/dL)', 'Hemoglobin A1c/Hemoglobin.total in _value (%)', 'Cholesterol in HDL [Mass/_value (mg/dL)', 
        'Insulin [Units/volume] in Serum o_value (ng/L)', 'Cholesterol in LDL [Mass/_value (mg/dL)', 'Natriuretic peptide.B prohormon_value (pg/mL)', 'Potassium [Moles/volume] in Ser_value (mEq/L)', 'Protein [Mass/volume] in Se_value (g/dL)', 'Sodium [Moles/volume] in Serum or _value (mEq/L)', 
        'Cholesterol [Mass/volum_value (mg/dL)', 'Triglyceride [Mass/volume] _value (mg/dL)', 'Troponin T.cardiac [Mass/volum_value (ng/L)', 'Albumin [Mass/volume] in Ur_value (_g/mL)', 'Creatinine [Mass/volume]_value (_g/mL)', 'Hemoglobin - g/dL', 'Hematocrit\xa0 - %', 'MCH - pg', 'MCHC - g/dL', 
        'MCV - fL', 'Platelets - x10E3/µL', 'Red Blood Cells (RBC) - x10E6/µL', 'RDW - %', 'White Blood Cells (WBC) - x10E3/µL', 'lettera_time', 'memory_trial1', 'memory_trial1_time', 'memory_trial2', 'memory_trial2_time', 'moca_abstraction', 'moca_abstraction_time', 'moca_combined_mis_score', 
        'moca_orientation', 'moca_orientation_time', 'moca_total_score', 'naming', 'naming_time', 'repetition', 'repetition_time', 'subtraction', 'subtraction_time', 'trails_visuospatial_executive', 'trails_visuospatial_executive_time', 'Waist Circumference (cm)', 'Weight (kilograms)', 
        'Waist to Hip Ratio (WHR)']

meas_new_df = []

counter = 0
for col_og in meas_cols:
    column = col_og.split('_value')[0]
    if 'Bilirubin' in column:
        column = 'Bilirubin, Total (mg/dL)'
    
    temp_colname = [x for x in real['measurement_source_value'].unique() if col_og.split('_value')[0] in x]
    if col_og == 'Creatinine [Mass/volume]_value (_g/mL)':
        column = 'import_urine_creatinine, Creatinine [Mass/volume]'
    
    elif len(temp_colname) > 1 and col_og not in ['Diastolic (mmHg)', 'Systolic (mmHg)']:
        
        if col_og == temp_colname[0]:
            column = temp_colname[0]
        
        elif col_og == temp_colname[1]:
            column = temp_colname[1]
        
        else:
            raise StopIteration()
    else:
        column = temp_colname[0]
    
    temp = real[real['measurement_source_value'] == column]
    print(f"{col_og}\t{column}:\t{len(temp)}")

    cols_same = {col:[] for col in temp.columns if len(temp[col].value_counts()) == 1 and not temp[col].isnull().mean() == 1}
    cols_nan = {col:[] for col in temp.columns if temp[col].isnull().mean() == 1}
    cols_new = {col:[] for col in temp.columns if len(temp[col].value_counts()) != 1 and ((not temp[col].isnull().mean() == 1) or col == 'value_source_value')}
    if 'operator_concept_id' not in cols_same.keys() and 'operator_concept_id' not in cols_nan.keys():
        cols_new['operator_concept_id'] = [4172703] * len(df)

    if 'measurement_type_concept_id' in cols_new:
        cols_new['measurement_type_concept_id'] = [37161934] * len(df)
    
    for i, row in df.iterrows():

        for col in cols_same:
            cols_same[col].append(temp.iloc[0][col])

        for col in cols_nan:
            cols_nan[col].append(np.nan)

        if 'measurement_id' in cols_new.keys():
            cols_new['measurement_id'].append(int(counter))
    
        cols_new['person_id'].append(int(row.name))

        now = datetime.datetime.now()
        omop_date = now.date()                     # date: YYYY-MM-DD
        omop_datetime = now.strftime('%Y-%m-%d %H:%M:%S')  # datetime: string format

        cols_new['measurement_date'].append(omop_date)
        cols_new['measurement_datetime'].append(omop_datetime)

        if 'visit_occurrence_id' in cols_new.keys():
            cols_new['visit_occurrence_id'].append(int(row.name))

        cols_new['value_as_number'].append(row[col_og])
        cols_new['value_source_value'].append(row[col_og])

        # High ranges vary by age for some measurements, but age isn't in this version of synthetic AI-READI
        # Will be fixed in future versions
        if 'range_high' in cols_new.keys():
            cols_new['range_high'].append(np.nan)
        
        if 'range_low' in cols_new.keys():
            cols_new['range_low'].append(np.nan)

        if 'unit_concept_id' in cols_new.keys():
            cols_new['unit_concept_id'].append(temp['unit_concept_id'].value_counts().sort_values().index[-1])

        counter += 1
    
    merged = {**cols_new, **cols_same, **cols_nan}
    meas_new_df.append(pd.DataFrame(merged))


BMI	bmi_vsorres, BMI:	2273
Diastolic (mmHg)	bp1_diabp_vsorres, Diastolic (mmHg):	2278
Systolic (mmHg)	bp1_sysbp_vsorres, Systolic (mmHg):	2278
clock_visuospatial_executive	clock_visuospatial_executive:	2266
clock_visuospatial_executive_time	clock_visuospatial_executive_time:	2207
cube_visuospatial_executive	cube_visuospatial_executive:	2267
cube_visuospatial_executive_time	cube_visuospatial_executive_time:	2207
delayed_recall_with_no_clue	delayed_recall_with_no_clue:	2266
delayed_recall_with_no_clue_time	delayed_recall_with_no_clue_time:	2207
digitspan	digitspan:	2265
digitspan_time	digitspan_time:	2207
Height (cm)	height_vsorres, Height (cm):	2273
Hip Circumference (cm)	hip_vsorres, Hip Circumference (cm):	2268
Albumin/Globulin ratio	import_a_g_ratio, Albumin/Globulin ratio:	2232
Albumin [Mass/volume] in Serum or_value (g/dL)	import_albumin, Albumin [Mass/volume] in Serum or:	2233
Alkaline phosphatase_value (IU/L)	import_alkaline_phosphatase, Alkaline phosphatase:	2233
Alanine aminotr

In [12]:
measurements_new = pd.concat(meas_new_df)[real.columns]
measurements_new.to_csv('/data/7TB/nick/synth_ai_readi_tabular/clinical_data/measurement.csv')
measurements_new

,measurement_id,person_id,measurement_concept_id,measurement_date,measurement_datetime,measurement_time,measurement_type_concept_id,operator_concept_id,value_as_number,value_as_concept_id,...,visit_detail_id,measurement_source_value,measurement_source_concept_id,unit_source_value,unit_source_concept_id,value_source_value,measurement_event_id,meas_event_field_concept_id,qualifier_concept_id,qualifier_source_value
0,0,0,4245997,2026-01-15,2026-01-15 14:16:14,,32862,0,18.405273,0,...,0,"bmi_vsorres, BMI",0,,0,NaN,0,0,0.0,
1,1,1,4245997,2026-01-15,2026-01-15 14:16:14,,32862,0,38.513360,0,...,0,"bmi_vsorres, BMI",0,,0,NaN,0,0,0.0,
2,2,2,4245997,2026-01-15,2026-01-15 14:16:14,,32862,0,33.701760,0,...,0,"bmi_vsorres, BMI",0,,0,NaN,0,0,0.0,
3,3,3,4245997,2026-01-15,2026-01-15 14:16:14,,32862,0,49.914143,0,...,0,"bmi_vsorres, BMI",0,,0,NaN,0,0,0.0,
4,4,4,4245997,2026-01-15,2026-01-15 14:16:14,,32862,0,29.160164,0,...,0,"bmi_vsorres, BMI",0,,0,NaN,0,0,0.0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10513,767809,10513,44809433,2026-01-15,2026-01-15 14:20:43,,32862,0,1.040242,0,...,0,"whr_vsorres, Waist to Hip Ratio (WHR)",0,,0,NaN,0,0,0.0,
10514,767810,10514,44809433,2026-01-15,2026-01-15 14:20:43,,32862,0,0.938202,0,...,0,"whr_vsorres, Waist to Hip Ratio (WHR)",0,,0,NaN,0,0,0.0,
10515,767811,10515,44809433,2026-01-15,2026-01-15 14:20:43,,32862,0,0.838912,0,...,0,"whr_vsorres, Waist to Hip Ratio (WHR)",0,,0,NaN,0,0,0.0,
10516,767812,10516,44809433,2026-01-15,2026-01-15 14:20:43,,32862,0,0.935838,0,...,0,"whr_vsorres, Waist to Hip Ratio (WHR)",0,,0,NaN,0,0,0.0,


# Wearables

NOTE: Wearable data are not the original time series format and full time-series EKGs are not available. Instead, features were manually extracted from the time series data of these modalities. The code we used to preprocess the real data and combine it with the clinical data (before using it to generate synthetic data) is available in 
- synth_ai_readi_tabular/1_get_wearable_data.py
- synth_ai_readi_tabular/2_aggregate_data_modalities.py
- synth_ai_readi_tabular/3_preprocess_aireadi.py

The .csv files created from these preprocessing scripts will match the structure of the synthetic data in synthetic_all.csv

In [13]:
cols = ['heart_rate_mean', 'blood_glucose_mean', 'resp_rate_mean', 'stress_mean', 'blood_glucose_std', 'total_steps',  'resting_heart_rate', 'total_kcal', 'sleep_light_hrs', 'sleep_deep_hrs', 'sleep_rem_hrs', 'sleep_awake_hrs', 'act_generic_hrs', 'act_running_hrs','act_walking_hrs', 'act_sedentary_hrs' ]
df[cols].reset_index().rename(columns={'index': 'participant_id'}).to_csv('/data/7TB/nick/synth_ai_readi_tabular/synthetic_wearable_activity_glucose.tsv', sep='\t')

## EKG

NOTE: Similarly to wearable data, synthetic time-series EKG data are not available. The files contain 0's for the actual waveform, but the synthetically generated measurements (normality, QT, P ... etc) are stored in these files.

In [14]:
real = pd.read_csv('/data/7TB/nick/ai_readi_v3/cardiac_ecg/manifest.tsv', delimiter='\t')


cols = ['Rate', 'PR', 'QRSD', 'QT', 'QTc', 'P', 'QRS', 'T']#, 'ABNORMAL', 'BORDERLINE', 'NORMAL', 'OTHERWISE NORMAL']

In [15]:
ekg_new = pd.DataFrame(np.nan, index=list(range(len(df))), columns=real.columns).astype('object')

# Set columns that do not change
cols_vals_same = [(c, d.value_counts().index[0]) for c,d in real.items() if c not in cols and 'wfdb' not in c]
for col, val in cols_vals_same:
    ekg_new.loc[:, col] = val

# Set participant_id and file columns
ekg_new.loc[:, 'participant_id'] = df.index.values
ekg_new.loc[:, 'wfdb_dat_filepath'] = [f'{i}_ecg_synthetic.dat' for i in range(len(df))]
ekg_new.loc[:, 'wfdb_hea_filepath'] = [f'{i}_ecg_synthetic.hea' for i in range(len(df))]

# Set synthetic columns
for col in cols: 
    ekg_new.loc[:, col] = df[col].copy()

ekg_new.to_csv('/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/manifest.tsv', sep='\t')

In [16]:
import wfdb
# Generate simplistic comments based on synthetic values
def generate_comments(rate, pr, qrsd, qt, qtc, p_wave, qrs, t_wave):
    comments = []

    # Comment 1: Rhythm
    comments.append('comment_1_key: Rhythm')
    comments.append(f'comment_1_val: Normal sinus rhythm at {rate} bpm')

    # Comment 2: PR Interval
    pr_text = 'within normal limits' if 120 <= pr <= 200 else 'abnormal'
    comments.append('comment_2_key: PR Interval')
    comments.append(f'comment_2_val: PR interval {pr_text} ({pr} ms)')

    # Comment 3: QRS Duration
    qrsd_text = 'normal' if qrsd <= 120 else 'prolonged'
    comments.append('comment_3_key: QRS Duration')
    comments.append(f'comment_3_val: QRS duration is {qrsd_text} ({qrsd} ms)')

    # Comment 4: QT Interval
    qt_text = 'within normal range' if qtc <= 450 else 'prolonged'
    comments.append('comment_4_key: QT Interval')
    comments.append(f'comment_4_val: QTc is {qtc} ms, which is {qt_text}')

    # Comment 5: Wave Morphology
    comments.append('comment_5_key: Wave Morphology')
    comments.append(f'comment_5_val: P wave: {p_wave}, QRS: {qrs}, T wave: {t_wave}')

    return comments



In [17]:
for i in range(len(df)):
    record = wfdb.rdrecord('/data/7TB/nick/ai_readi_v3/'+real.loc[0, 'wfdb_hea_filepath'][0:-4])

    new_comments = []
    for line in record.comments:
        k,v = line.split(': ')
        if k in cols:
            v = df.loc[i, k]

        elif k == 'participant_id':
            v = i
        
        elif k == 'participant_position':
            v = '0 degrees (supine)'

        elif k == 'interpretation_comment_2':
            interp = ['ABNORMAL', 'BORDERLINE', 'NORMAL', 'OTHERWISE NORMAL'][df.loc[i][['ABNORMAL', 'BORDERLINE', 'NORMAL', 'OTHERWISE NORMAL']].argmax()]
            v = f'- {interp} -'
            new_comments.append(": ".join([k,str(v)]))
            break

        new_comments.append(": ".join([k,str(v)]))

    new_comments = new_comments + generate_comments(df.loc[i, 'Rate'], df.loc[i, 'PR'] , df.loc[i, 'QRSD'], df.loc[i: 'QT'], df.loc[i, 'QTc'], df.loc[i, 'P'], df.loc[i, 'QRS'], df.loc[i, 'T'])

    record.p_signal[:] = 0
    # Save the modified signal and header
    savedir = f'/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/{i}'
    os.makedirs(savedir, exist_ok=True)
    os.chdir(savedir)
    wfdb.wrsamp(
        record_name=f'{i}_ecg_synthetic',
        fs=record.fs,
        units=record.units,
        sig_name=record.sig_name,
        p_signal=record.p_signal,
        fmt=['16'] * record.n_sig,  # keep same format
        comments=new_comments
    )

    print(savedir)

/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/0
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/1
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/2
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/3
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/4
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/5
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/6
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/7
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/8
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/9
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/10
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/11
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/12
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/13
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/14
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/15
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/16
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/17
/data/7TB/nick/synth_ai_readi_tabular/cardiac_ecg/18
/da